# Two-tower dengan gerbang personal (PyTorch)

[![Buka di Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/imamrozi/recsys-hibrida-multisinyal-notebooks/blob/main/notebooks/bab05_emosi_sesi/03_two_tower_gerbang_personal.ipynb)

**Buku pendamping:** *Sistem Rekomendasi Hibrida Multi-Sinyal: Rating, Sentimen, dan Emosi* · Bab 5 · Kode 5.4.1

**Tujuan belajar**

- Membaca arsitektur two-tower dengan gerbang vektor antara preferensi jangka panjang dan sinyal sesi.
- Melatih dengan loss softmax in-batch + koreksi logQ + regularisasi entropi gerbang.
- Melihat apakah gerbang yang dipelajari mengikuti struktur data simulasi.

> **Catatan jujur.** Semua data di notebook ini **simulasi** dan dibuat agar mudah dijalankan (CPU, hitungan detik). Angka yang keluar menunjukkan *perilaku metode*, bukan hasil empiris pada dataset nyata, dan tidak boleh dikutip sebagai bukti. Kode bersifat ilustratif dan **belum production-ready** (tanpa validasi masukan, tanpa uji beban).

> **Peringatan.** Notebook ini membutuhkan PyTorch (sudah tersedia di Google Colab). Hasil asersi di bawah dirancang untuk simulasi kecil di CPU, tetapi tolok ukur kuantitatifnya dikonfirmasi lewat eksekusi otomatis di GitHub Actions, bukan dijamin lintas versi PyTorch. Kelompok pengguna 'konsisten' vs 'pengikut sesi' dalam simulasi adalah **asumsi**.

## 1. Kode dari buku

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(0); np.random.seed(0)


class ItemTower(nn.Module):
    def __init__(self, n_items, n_feat, d=64):
        super().__init__()
        self.emb = nn.Embedding(n_items, d)
        self.mlp = nn.Sequential(nn.Linear(d + n_feat + 2, d), nn.ReLU(), nn.Linear(d, d))
    def forward(self, item_ids, feats, va):            # va: (n, 2) = v_i
        x = torch.cat([self.emb(item_ids), feats, va], dim=-1)
        return F.normalize(self.mlp(x), dim=-1)

class GatedUserTower(nn.Module):
    def __init__(self, n_users, n_ctx, d=64, eta=3.0):
        super().__init__()
        self.user = nn.Embedding(n_users, d)
        self.long = nn.Sequential(nn.Linear(d, d), nn.ReLU(), nn.Linear(d, d))
        self.sess = nn.Linear(d, d)
        self.gate = nn.Linear(d + n_ctx + d, d)       # Pers. (5-7), gerbang bervektor
        self.eta = eta
    def forward(self, user_ids, ctx, sess_item_vecs, sess_mask):
        e_u = self.user(user_ids)
        h_L = self.long(e_u)
        T = sess_item_vecs.size(1)                      # (B, T, d), item terbaru di akhir
        age = torch.arange(T - 1, -1, -1, device=e_u.device).float()
        w = torch.exp(-age / self.eta) * sess_mask      # bobot kebaruan (Tabel 5.3)
        w = w / w.sum(1, keepdim=True).clamp_min(1e-9)
        h_S = self.sess((w.unsqueeze(-1) * sess_item_vecs).sum(1))
        g = torch.sigmoid(self.gate(torch.cat([e_u, ctx, h_S], dim=-1)))
        h_u = g * h_L + (1 - g) * h_S                   # Pers. (5-8)
        return F.normalize(h_u, dim=-1), g

def loss_fn(h_u, h_i, log_q, g, tau=0.05, beta_g=0.01):
    """h_u, h_i: (B, d) pasangan positif sejajar; log_q: (B,) log frekuensi item."""
    logits = h_u @ h_i.T / tau - log_q.unsqueeze(0)    # Pers. (5-5) + koreksi logQ
    labels = torch.arange(h_u.size(0), device=h_u.device)
    l_rec = F.cross_entropy(logits, labels)             # Pers. (5-6)
    g_bar = g.mean(0).clamp(1e-6, 1 - 1e-6)
    omega = (g_bar * g_bar.log() + (1 - g_bar) * (1 - g_bar).log()).sum()  # Pers. (5-10)
    return l_rec + beta_g * omega

## 2. Data simulasi

Dunia kecil dengan 300 item di 6 kelompok selera (cluster) dan koordinat valence–arousal per item. Dua tipe pengguna:

- **Konsisten** (50%): item berikutnya diambil dari cluster favorit tetap, apa pun sesinya. Sinyal sesi tidak membantu.
- **Pengikut sesi** (50%): item berikutnya mengikuti cluster item terakhir di sesi. Preferensi jangka panjang tidak membantu.

Model **tidak diberi tahu** tipe pengguna. Jika arsitekturnya masuk akal, gerbang `g` (bobot jangka panjang) seharusnya lebih besar untuk pengguna konsisten.

In [ ]:
N_ITEM, N_USER, N_CL, T, D = 300, 400, 6, 6, 32
rng = np.random.default_rng(0)
item_cl = rng.integers(0, N_CL, N_ITEM)
items_by_cl = [np.where(item_cl == c)[0] for c in range(N_CL)]
va = np.clip(rng.normal(0, .5, (N_CL, 2))[item_cl] + rng.normal(0, .1, (N_ITEM, 2)), -1, 1).astype("float32")
feats = np.eye(N_CL, dtype="float32")[item_cl]                      # fitur item: one-hot cluster
fav = rng.integers(0, N_CL, N_USER)
konsisten = rng.random(N_USER) < 0.5

def contoh(n):
    u = rng.integers(0, N_USER, n)
    sess = np.zeros((n, T), int); pos = np.zeros(n, int)
    for k in range(n):
        cl_s = rng.integers(0, N_CL, T)
        sess[k] = [rng.choice(items_by_cl[c]) for c in cl_s]
        cl_t = fav[u[k]] if konsisten[u[k]] else cl_s[-1]
        pos[k] = rng.choice(items_by_cl[cl_t])
    return u, sess, pos

u_tr, s_tr, p_tr = contoh(6000)
u_te, s_te, p_te = contoh(1000)
freq = np.bincount(p_tr, minlength=N_ITEM) + 1.0
log_q_all = torch.tensor(np.log(freq / freq.sum()), dtype=torch.float32)
print("contoh latih:", len(u_tr), "| porsi pengguna konsisten:", konsisten.mean().round(2))

## 3. Pelatihan

In [ ]:
item_ids_all = torch.arange(N_ITEM)
F_all, VA_all = torch.tensor(feats), torch.tensor(va)

def latih(epochs=15, bs=256, beta_g=0.01, lr=3e-3, seed=0):
    torch.manual_seed(seed)
    it = ItemTower(N_ITEM, N_CL, D)
    ut = GatedUserTower(N_USER, 1, D)
    opt = torch.optim.Adam(list(it.parameters()) + list(ut.parameters()), lr=lr)
    ctx_dummy = lambda n: torch.zeros(n, 1)
    riw = []
    for ep in range(epochs):
        perm = np.random.default_rng(seed + ep).permutation(len(u_tr))
        tot = 0
        for b in range(0, len(perm), bs):
            ix = perm[b:b + bs]
            u = torch.tensor(u_tr[ix]); s = torch.tensor(s_tr[ix]); p = torch.tensor(p_tr[ix])
            sv = it(s.reshape(-1), F_all[s.reshape(-1)], VA_all[s.reshape(-1)]).reshape(len(ix), T, D)
            h_u, g = ut(u, ctx_dummy(len(ix)), sv, torch.ones(len(ix), T))
            h_i = it(p, F_all[p], VA_all[p])
            loss = loss_fn(h_u, h_i, log_q_all[p], g, beta_g=beta_g)
            opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item() * len(ix)
        riw.append(tot / len(perm))
    return it, ut, riw

@torch.no_grad()
def evaluasi(it, ut, k=10):
    u = torch.tensor(u_te); s = torch.tensor(s_te); p = torch.tensor(p_te)
    sv = it(s.reshape(-1), F_all[s.reshape(-1)], VA_all[s.reshape(-1)]).reshape(len(u), T, D)
    h_u, g = ut(u, torch.zeros(len(u), 1), sv, torch.ones(len(u), T))
    h_all = it(item_ids_all, F_all, VA_all)
    top = (h_u @ h_all.T).topk(k, dim=1).indices
    hit = (top == p.unsqueeze(1)).any(1).float().numpy()
    return hit.mean(), g.mean(1).numpy()

it, ut, riw = latih()
hit, g_rata = evaluasi(it, ut)
g_kons = g_rata[konsisten[u_te]].mean(); g_sess = g_rata[~konsisten[u_te]].mean()
acak = 10 / N_ITEM
print(f"loss awal -> akhir: {riw[0]:.3f} -> {riw[-1]:.3f}")
print(f"hit@10 = {hit:.3f} (acak = {acak:.3f})")
print(f"rata-rata gerbang jangka panjang: konsisten {g_kons:.3f} | pengikut sesi {g_sess:.3f}")

fig, ax = plt.subplots(1, 2, figsize=(8, 3))
ax[0].plot(riw, marker="o"); ax[0].set(xlabel="epoch", ylabel="loss", title="Kurva latih")
ax[1].hist(g_rata[konsisten[u_te]], bins=20, alpha=.6, label="konsisten")
ax[1].hist(g_rata[~konsisten[u_te]], bins=20, alpha=.6, label="pengikut sesi")
ax[1].set(xlabel="rata-rata g (bobot jangka panjang)", title="Gerbang per tipe"); ax[1].legend()
fig.tight_layout()

assert riw[-1] < riw[0]
assert hit > 3 * acak

**Cara membaca hasil.** Pertama, loss turun dan hit@10 jauh di atas acak: model mempelajari struktur cluster. Kedua, bandingkan rata-rata gerbang antar tipe pengguna. Jika gerbang memang menangkap struktur data, pengguna konsisten bernilai `g` lebih tinggi. Hasil ini **tidak dijamin** pada setiap seed dan setiap versi PyTorch; periksa angka yang keluar, dan lihat ablasi di bawah sebelum menarik kesimpulan.

## 4. Ablasi: apakah gerbang membantu?

Bandingkan dengan dua pembanding degenerat yang dibuat dengan membekukan gerbang: hanya jangka panjang (`g = 1`) dan hanya sesi (`g = 0`), lewat subclass sederhana. Ini menjawab pertanyaan inti: pada data campuran, apakah gerbang yang dipelajari mengalahkan kedua ekstrem?

In [ ]:
class GerbangTetap(GatedUserTower):
    def __init__(self, *a, nilai=1.0, **k):
        super().__init__(*a, **k); self.nilai = nilai
    def forward(self, user_ids, ctx, sess_item_vecs, sess_mask):
        e_u = self.user(user_ids); h_L = self.long(e_u)
        T_ = sess_item_vecs.size(1)
        age = torch.arange(T_ - 1, -1, -1).float()
        w = torch.exp(-age / self.eta) * sess_mask
        w = w / w.sum(1, keepdim=True).clamp_min(1e-9)
        h_S = self.sess((w.unsqueeze(-1) * sess_item_vecs).sum(1))
        g = torch.full_like(h_L, self.nilai)
        return F.normalize(g * h_L + (1 - g) * h_S, dim=-1), g

def latih_ut(kelas, seed=0, epochs=15, **kw):
    torch.manual_seed(seed)
    it_ = ItemTower(N_ITEM, N_CL, D); ut_ = kelas(N_USER, 1, D, **kw)
    opt = torch.optim.Adam(list(it_.parameters()) + list(ut_.parameters()), lr=3e-3)
    for ep in range(epochs):
        perm = np.random.default_rng(seed + ep).permutation(len(u_tr))
        for b in range(0, len(perm), 256):
            ix = perm[b:b + 256]
            u = torch.tensor(u_tr[ix]); s = torch.tensor(s_tr[ix]); p = torch.tensor(p_tr[ix])
            sv = it_(s.reshape(-1), F_all[s.reshape(-1)], VA_all[s.reshape(-1)]).reshape(len(ix), T, D)
            h_u, g = ut_(u, torch.zeros(len(ix), 1), sv, torch.ones(len(ix), T))
            loss = loss_fn(h_u, it_(p, F_all[p], VA_all[p]), log_q_all[p], g, beta_g=0.0)
            opt.zero_grad(); loss.backward(); opt.step()
    return it_, ut_

hasil = {}
for nama, kelas, kw in (("gerbang dipelajari", GatedUserTower, {}),
                        ("hanya jangka panjang (g=1)", GerbangTetap, {"nilai": 1.0}),
                        ("hanya sesi (g=0)", GerbangTetap, {"nilai": 0.0})):
    a, b = latih_ut(kelas, **kw)
    hasil[nama] = evaluasi(a, b)[0]
for k, v in hasil.items():
    print(f"{k:30s} hit@10 = {v:.3f}")
assert hasil["gerbang dipelajari"] > hasil["hanya jangka panjang (g=1)"] - 0.02

**Interpretasi (perkiraan; periksa terhadap keluaran Anda).** Pada data campuran, kedua ekstrem masing-masing gagal pada separuh pengguna, sehingga gerbang yang dipelajari seharusnya setidaknya sebanding dengan yang terbaik dari keduanya dan idealnya lebih tinggi. Asersi di atas sengaja longgar (toleransi 0,02) karena simulasi kecil dan satu seed. Jika gerbang tidak unggul pada eksekusi Anda, itu informasi yang sah: tambah epoch, ubah `eta`, atau jalankan beberapa seed sebelum menyimpulkan.

## Pertanyaan refleksi

1. Mengapa koreksi logQ perlu pada negatif in-batch? Apa yang terjadi bila item populer terlalu sering menjadi negatif?
2. Setel `beta_g` ke 0, 0,01, dan 1. Bagaimana distribusi gerbang berubah? Apakah entropi tinggi atau rendah yang diharapkan?
3. Tambahkan konteks nyata (misalnya jam hari) ke `ctx` yang sekarang nol. Perubahan apa yang Anda harapkan pada gerbang?